In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42

# COFIGURACIÓN GENERAL
ROOT_PATH = r'C:\Users\AaronMCC\Documents\arxiv_cs_dataset_exploration'

# --- Rutas de entrada ---
DATA_PATH_TOP_10     = ROOT_PATH + r'dataset\arxiv_cs_top10.parquet'

# --- Rutas de salida ---
DATA_PATH_TOP_10_LIMPIO_BAELINE = ROOT_PATH + r'dataset\arxiv_cs_top10_limpieza_baseline.parquet'
DIR_MODELS = ROOT_PATH + r'models\'
DIR_RESULTADOS = ROOT_PATH + r'resultados\'

os.mkdirs(DIR_MODELS, exist_ok = True)
os.mkdirs(DIR_RESULTADOS, exist_ok = True)


# now = datetime.now() # Obtener la fecha y hora actual
# date_time = now.strftime("%d_%m_%Y_%H_%M_%S")

# OUTPUT_DIR        = r'C:\Users\AaronMCC\Documents\Rest-Mex2025\resultados\res_' + date_time

# os.mkdir(OUTPUT_DIR)

# BEST_MODEL_PATH   = r'C:\Users\AaronMCC\Documents\Rest-Mex2025\models\best_model_' + date_time + '.pt'
# CSV_HISTORY       = OUTPUT_DIR + r'\training_history_' + date_time + '.csv'
# CSV_OPTUNA        = OUTPUT_DIR + r'\optuna_results_' + date_time + '.csv'
# PLOT_CURVES       = OUTPUT_DIR + r'\curvas_entrenamiento_' + date_time + '.png'
# PLOT_CONFUSION    = OUTPUT_DIR + r'\confusion_matrix_' + date_time + '.png'





## Preprocesamiento de texto
Limpiar los abstracts antes de vectorizar:

- [x] Convertir a minúsculas.
- [x] Eliminar caracteres especiales, números sueltos y puntuación.
- [x] Eliminar stopwords en inglés.
- [x] Aplicar lematización 



In [ ]:
# Carga del Dataset
df_top10 = pd.read_parquet(DATA_PATH_TOP_10)
print("Dataset cargado: OK!")


def limpieza_baseline(text):
    if not isinstance(text, str) or len(text) < 5:
        return ""
    text = text.lower()
    # URLs y correos
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S*@\S*\s?', '', text)
    # Eliminación de fórmulas matemáticas en LaTeX
    text = re.sub(r'\$.*?\$', '', text)
    # Eliminación de comandos y macros de LaTeX
    text = re.sub(r'\\[a-zA-Z]+\{.*?\}', '', text)
    # Solo letras y espacios
    text = re.sub(r'[^a-z\s]', '', text)
    # Normalización de espacios
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Iniciando limpieza... Puede demorar unos minutos")

# Aplicar limpieza sobre el abstract
df_top10['texto_limpio'] = df_top10['abstract'].progress_apply(limpieza_baseline)

# Conservar solo columnas relevantes
df_top10 = df_top10[['title', 'abstract', 'cs_primary', 'texto_limpio']]

# Guardar el DF
df_top10.to_parquet(DATA_PATH_TOP_10_LIMPIO_BAELINE)
print("Limpieza completada y guardada: OK!")

## Entranamiento

In [ ]:
# Carga del corpus limpio 
df = pd.read_parquet(DATA_PATH_TOP_10_LIMPIO_BAELINE)
print(f"Registros cargados : {len(df):,}")
print(f"Clases disponibles : {df['cs_primary'].nunique()}")
print(df['cs_primary'].value_counts())

In [ ]:
# División train / validation 
X = df['texto_limpio'].fillna("")
y = df['cs_primary']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"Train : {len(X_train):,} documentos")
print(f"Val   : {len(X_val):,} documentos")

In [ ]:
# Pesos de clase calculados sobre el conjunto de entrenamiento
clases      = y_train.unique()
K           = len(clases)
N_total     = len(y_train)
conteos     = y_train.value_counts()

pesos = {clase: N_total / (K * conteos[clase]) for clase in clases}

print("Pesos de clase:")
for clase, peso in sorted(pesos.items(), key=lambda x: x[1], reverse=True):
    print(f"  {clase:10s}  n={conteos[clase]:>7,}  peso={peso:.4f}")

In [ ]:
# Vectorización TF-IDF 
vectorizer = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),       # unigramas + bigramas
    sublinear_tf=True,        # escala logarítmica en frecuencias
    min_df=2,                 # ignorar términos en menos de 2 docs
    stop_words='english'      # stopwords 
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)

print(f"Dimensión matriz TF-IDF train : {X_train_tfidf.shape}")
print(f"Dimensión matriz TF-IDF val   : {X_val_tfidf.shape}")

In [ ]:
# Entrenamiento Regresión Logística
model = LogisticRegression(
    class_weight=pesos,    # compensar desbalance
    max_iter=1000,
    C=1.0,
    solver='saga',         # eficiente para corpus grandes
    random_state=RANDOM_SEED,
    n_jobs=-1              # usar todos los núcleos disponibles
)

print("Entrenando modelo...")
model.fit(X_train_tfidf, y_train)
print("Entrenamiento completado.")

In [ ]:
# Evaluación interna sobre validación arXiv 
y_pred_val = model.predict(X_val_tfidf)

print("=" * 60)
print("EVALUACIÓN INTERNA — Validación arXiv (20%)")
print("=" * 60)
print(classification_report(y_val, y_pred_val, digits=4))

macro_f1_val = f1_score(y_val, y_pred_val, average='macro')
print(f"Macro F1 validación arXiv : {macro_f1_val:.4f}")

In [ ]:
# Evaluación transferencia pura sobre UAM-A 
# Cargar el dataset UAM-A
df_uam = pd.read_parquet('arxiv_cs.parquet')   

# Aplicar la misma limpieza ligera usada en el corpus arXiv
df_uam['texto_limpio'] = df_uam['abstract'].apply(limpieza_baseline)

# Vectorizar con el TF-IDF 
X_uam_tfidf = vectorizer.transform(df_uam['texto_limpio'].fillna(""))

print(f"Documentos UAM-A vectorizados: {X_uam_tfidf.shape[0]:,}")

In [ ]:
# Predicciones sobre UAM-A
y_pred_uam = model.predict(X_uam_tfidf)

# Guardar predicciones en el DataFrame
df_uam['pred_baseline'] = y_pred_uam

print("Distribución de predicciones sobre UAM-A:")
print(pd.Series(y_pred_uam).value_counts().to_string())

In [ ]:
# Métricas UAM-A 

# y_true_uam = df_uam['etiqueta_manual']
# print("=" * 60)
# print("EVALUACIÓN TRANSFERENCIA PURA — UAM-A (733 docs)")
# print("=" * 60)
# print(classification_report(y_true_uam, y_pred_uam, digits=4))
# macro_f1_uam = f1_score(y_true_uam, y_pred_uam, average='macro')
# print(f"Macro F1 UAM-A : {macro_f1_uam:.4f}")

# Solo la distribución de predicciones
fig, ax = plt.subplots(figsize=(10, 5))
pd.Series(y_pred_uam).value_counts().sort_values().plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Distribución de predicciones baseline — UAM-A", fontsize=13)
ax.set_xlabel("Documentos")
plt.tight_layout()
plt.savefig(DIR_RESULTADOS + "pred_baseline_uam.png", dpi=150)
plt.show()

In [ ]:
# ─── Guardar modelo y vectorizador para comparación posterior ─────────────────
import joblib

joblib.dump(model,      DIR_MODELS + r'\models\baseline_logreg.pkl')
joblib.dump(vectorizer, DIR_MODELS + r'\models\baseline_tfidf.pkl')
print("Modelo y vectorizador guardados.")